# Prepare Datasets for Machine Learning

In previous notebooks, SNOTEL, PRISM, and DEM datasets were downloaded and cleaned in preparation for application to machine learning. This notebook will perform final cleaning and calculate some further statistics for the PRISM dataset; the following notebook will perform the machine learning.

## Step 1: Import Libraries and Set Up Project Directory

In [ ]:
# import libraries

# file management
import os
import pathlib
from pathlib import Path
import sys

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

KeyboardInterrupt: 

In [ ]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Station Filtering

This step will filter the SNOTEL stations used in the ML model to stations with at least 20 years of record keeping. Ideally, stations with 30 years would be used. However, this will likely disqualify too many stations and reduce the dataset size too much.

In [ ]:
# Import PRISM dataset

# set a path
prism_path = Path(cleaned_data_dir, 'prism', 'prism_mhw_1990_2020_cleaned_with_crs.nc')

# open prism dataset
prism_ds = xr.open_dataset(prism_path, decode_coords='all')
print(prism_ds.rio.crs)

In [ ]:
# Import SNOTEL dataset

# set a path
snotel_path = Path(cleaned_data_dir, 'bcqc_snotel_1990-2020_RAW.nc')

# open SNOTEL dataset
snotel_ds = xr.open_dataset(snotel_path, decode_coords='all')
print(snotel_ds.rio.crs)

In [ ]:
# extract prism values at station locations

# grab station metadata
stations = snotel_ds.get_stations

# initialize list
prism_filtered_list = []

# filter through prism and extract timeseries for each pixel that has a snotel station in it
for station in stations:
    st_id = station.id
    lat = station.lat
    lon = station.lon

    # filter prism dataset
    prism_pixel = prism_ds.sel(
        # grab pixel that matches station coords
        lon = lon, lat = lat,
        # get the pixel nearest to the station
        method='nearest'
        # convert to DF
        ).to_dataframe.reset_index()
    
    # keep only climate and time data

    # append to list
    prism_filtered_list.append(prism_pixel)


## Step 3: Geospatial Joins

This step will join the PRISM, SNOTEL, and DEM datasets into a single dataframe.

## Step 4: NaN Removal

This step will remove any remaining NaNs from the PRISM and SNOTEL datasets using interpolation or regression.

## Step 5: Statistics Calculations

This step will calculate additional statistics on the PRISM and SNOTEL stations, such as rolling 7-day averages of temperature and precipitation (cite here), as well as cumulative SWE calculations to keep track of snowpack.

## Step 6: Train/Test Splitting

This step will perform train/test splitting for use in the machine learning model. Because the data is so temporal in nature, the train/test split will be performed by removing the last _fill in_ years of the dataset. 

## Step 7: Save dataset for use in the model